# Notebook 1 — Modèles de friction : Coulomb et RSF Scholz

Comparaison de trois modèles de friction sur un système 1-bloc 1D avec σ_n imposé (constant) :
- **Coulomb** : µ_s / µ_d constants, seuil discret stick → slip
- **RSF Scholz (Euler)** : loi taux-et-état, intégration Euler explicite
- **RSF Scholz (RK45)** : même loi, intégrateur adaptatif Dormand–Prince

Référence : Erickson et al. (2011), éq. (4)


In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import gc
import os

# ═══════════════════════════════════════════════════════════════════════════
#  RK45 INTEGRATOR (shared by all codes)
# ═══════════════════════════════════════════════════════════════════════════

_A_tab = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [1/5, 0, 0, 0, 0, 0, 0],
    [3/40, 9/40, 0, 0, 0, 0, 0],
    [44/45, -56/15, 32/9, 0, 0, 0, 0],
    [19372/6561, -25360/2187, 64448/6561, -212/729, 0, 0, 0],
    [9017/3168, -355/33, 46732/5247, 49/176, -5103/18656, 0, 0],
    [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0],
])
_B5 = np.array([35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0])
_B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40])
_C  = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1])

def rk45_step(f, t, y, h):
    K = np.zeros((7, len(y)))
    K[0] = f(t, y)
    for i in range(1, 7):
        yi = y + h * np.dot(_A_tab[i, :i], K[:i])
        K[i] = f(t + _C[i] * h, yi)
    y5 = y + h * np.dot(_B5, K)
    y4 = y + h * np.dot(_B4, K)
    return y5, np.abs(y5 - y4)

def rk45_integrate(f, t_span, y0, rtol=1e-8, atol=1e-10,
                   h_init=1e-3, h_min=1e-12, h_max=10.0,
                   max_steps=2_000_000):
    t0, tf = t_span
    y = np.array(y0, dtype=float)
    t = t0
    h = min(h_init, tf - t0)
    chunk = 100_000
    t_list = np.zeros(chunk)
    y_list = np.zeros((chunk, len(y)))
    t_list[0] = t; y_list[0] = y
    idx = 1; n_alloc = chunk; safety = 0.9; n_steps = 0; n_reject = 0

    while t < tf:
        if n_steps >= max_steps:
            print(f"  WARNING: hit max_steps={max_steps:,}"); break
        h = min(h, tf - t)
        h = max(h, h_min)
        y_new, err = rk45_step(f, t, y, h)
        scale = atol + rtol * np.maximum(np.abs(y), np.abs(y_new))
        err_norm = np.sqrt(np.mean((err / scale)**2))
        if err_norm <= 1.0:
            t += h; y = y_new; n_steps += 1
            if idx >= n_alloc:
                t_list = np.concatenate([t_list, np.zeros(chunk)])
                y_list = np.concatenate([y_list, np.zeros((chunk, len(y)))])
                n_alloc += chunk
            t_list[idx] = t; y_list[idx] = y; idx += 1
            h *= (5.0 if err_norm < 1e-30 else safety * err_norm**(-0.2))
            h = min(h, h_max)
        else:
            h *= safety * err_norm**(-0.25)
            h = max(h, h_min); n_reject += 1
    print(f"  {idx:,} steps ({n_reject:,} rejected)")
    return t_list[:idx], y_list[:idx]

## Intégrateur RK45 adaptatif (Dormand–Prince) — partagé par tous les codes

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
#  SHARED MODELS
# ═══════════════════════════════════════════════════════════════════════════

def coulomb_sigma_n(mu_s, mu_d, sigma_n, m, k, v_load, dt, duration):
    timesteps = np.arange(0, duration + dt, dt)
    n = len(timesteps)
    x = np.zeros(n); v = np.zeros(n)
    Fs = np.zeros(n); Ff = np.zeros(n)
    stick = np.ones(n, dtype=bool)
    for i in range(1, n):
        x_load = v_load * timesteps[i]
        Fs[i] = k * (x_load - x[i-1])
        if stick[i-1]:
            Fmax = mu_s * sigma_n
            Ff[i] = np.clip(Fs[i], -Fmax, Fmax)
            if Fs[i] > Fmax:
                stick[i] = False; v[i] = 1e-8; x[i] = x[i-1]
                Ff[i] = mu_d * sigma_n
            else:
                x[i] = x[i-1]
        else:
            if v[i-1] <= 0:
                v[i] = 0.0; stick[i] = True; x[i] = x[i-1]
            else:
                Ff[i] = mu_d * sigma_n
                a = (Fs[i] - Ff[i]) / m
                v[i] = v[i-1] + a * dt
                x[i] = x[i-1] + v[i-1] * dt + 0.5 * a * dt**2
                if v[i] <= 0: v[i] = 0.0; stick[i] = True
    return timesteps, x, v, Fs, Ff


def rsf_scholz_euler(mu0, a, b, v0, Dc, sigma_n, m, k, v_load, dt, duration):
    timesteps = np.arange(0, duration + dt, dt)
    n = len(timesteps)
    x = np.zeros(n); v = np.zeros(n); theta = np.zeros(n)
    Fs = np.zeros(n); Ff = np.zeros(n)
    stick = np.ones(n, dtype=bool); theta[0] = Dc / v0
    for i in range(1, n):
        x_load = v_load * timesteps[i]
        Fs[i] = k * (x_load - x[i-1])
        if stick[i-1]:
            mu_st = mu0 + b * np.log(theta[i-1] * v0 / Dc)
            Fmax = mu_st * sigma_n
            Ff[i] = np.clip(Fs[i], -Fmax, Fmax)
            if np.abs(Fs[i]) > Fmax:
                stick[i] = False; v[i] = 1e-8; x[i] = x[i-1]
                theta[i] = Dc / 1e-8
            else:
                x[i] = x[i-1]; theta[i] = theta[i-1] + dt
        else:
            vv = max(v[i-1], 1e-12)
            mu = mu0 + a * np.log(1 + vv/v0) + b * np.log(theta[i-1]*v0/Dc)
            Ff[i] = mu * sigma_n * np.sign(v[i-1])
            acc = (Fs[i] - Ff[i]) / m
            v[i] = v[i-1] + acc * dt
            x[i] = x[i-1] + v[i-1] * dt
            theta[i] = theta[i-1] + (1 - v[i-1]*theta[i-1]/Dc) * dt
            if np.abs(v[i]) <= 1e-10:
                v[i] = 0.0; stick[i] = True; theta[i] = theta[i-1]
    return timesteps, x, v, Fs, Ff, theta


def rsf_scholz_verlet(mu0, a, b, v0, Dc, sigma_n, m, k, v_load, dt, duration):
    timesteps = np.arange(0, duration + dt, dt)
    n = len(timesteps)
    x = np.zeros(n); v = np.zeros(n); theta = np.zeros(n)
    Fs = np.zeros(n); Ff = np.zeros(n)
    stick = np.ones(n, dtype=bool); theta[0] = Dc / v0

    def fric_force(vv, th):
        ve = max(abs(vv), 1e-12)
        mu = mu0 + a*np.log(1+ve/v0) + b*np.log(th*v0/Dc)
        return mu * sigma_n * (1.0 if vv >= 0 else -1.0)

    for i in range(1, n):
        x_load = v_load * timesteps[i]
        if stick[i-1]:
            Fs[i] = k * (x_load - x[i-1])
            mu_st = mu0 + b * np.log(theta[i-1]*v0/Dc)
            Fmax = mu_st * sigma_n
            Ff[i] = np.clip(Fs[i], -Fmax, Fmax)
            if np.abs(Fs[i]) > Fmax:
                stick[i] = False; v[i] = 1e-8; x[i] = x[i-1]
                theta[i] = Dc / 1e-8
            else:
                x[i] = x[i-1]; theta[i] = theta[i-1] + dt
        else:
            Fs_old = k * (v_load * timesteps[i-1] - x[i-1])
            Ff_old = fric_force(v[i-1], theta[i-1])
            a_old = (Fs_old - Ff_old) / m
            x[i] = x[i-1] + v[i-1]*dt + 0.5*a_old*dt**2
            theta[i] = theta[i-1] + (1 - v[i-1]*theta[i-1]/Dc)*dt
            Fs_new = k * (v_load * timesteps[i] - x[i])
            v_half = v[i-1] + 0.5*a_old*dt
            Ff_new = fric_force(v_half, theta[i])
            a_new = (Fs_new - Ff_new) / m
            v[i] = v[i-1] + 0.5*(a_old + a_new)*dt
            Fs[i] = Fs_new; Ff[i] = Ff_new
            if np.abs(v[i]) <= 1e-10:
                v[i] = 0.0; stick[i] = True; theta[i] = theta[i-1]
    return timesteps, x, v, Fs, Ff, theta


def make_rhs_sigma_n(mu0, a, b, v0, v_slider, Dc, M, k, sigma_n):
    def rhs(t, y):
        u, vel, Theta = y
        v_safe = np.sign(vel)*max(abs(vel), 1e-30)
        ln_v = np.log(abs(v_safe)/v0)
        du     = vel - v_slider
        dv     = (-1.0/M)*(k*u + sigma_n*(mu0 + Theta + a*ln_v))
        dTheta = -(v_safe/Dc)*(Theta + b*ln_v)
        return np.array([du, dv, dTheta])
    return rhs

def simulate_rk45_sigma_n(mu0, a, b, v0, v_slider, Dc, M, k, sigma_n,
                          duration, rtol=1e-8, atol=1e-10, h_max=1.0):
    rhs = make_rhs_sigma_n(mu0, a, b, v0, v_slider, Dc, M, k, sigma_n)
    u0 = -sigma_n*mu0/k + Dc
    t, y = rk45_integrate(rhs, [0, duration], [u0, v0, 0.0],
                          rtol=rtol, atol=atol, h_max=h_max)
    x_block = y[:,0] + v_slider*t
    theta = (Dc/v0)*np.exp(y[:,2]/b)
    return t, x_block, y[:,1], theta

## Modèles de friction : Coulomb, RSF Euler, RSF Verlet, RSF RK45

In [3]:
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║  CODE 1 : sigma_n = cte — comparaison 3 modeles                         ║
# ╚═══════════════════════════════════════════════════════════════════════════╝

def run_code1():
    print("="*70)
    print("  CODE 1 : Comparaison sigma_n = m*g  vs  sigma_n = cte")
    print("="*70)

    m = 1.0; g = 9.81; k = 1000; v_load = 0.001
    dt = 1e-4; duration = 10
    sigma_mg = m*g
    sigma_imp = 50.0

    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle(r'$\sigma_n = mg$ (9.81 N)  vs  $\sigma_n^{imp}$ (50 N)',
                  fontsize=14, fontweight='bold')

    # --- Coulomb ---
    print("\n  Coulomb...")
    for col, (sn, title) in enumerate([(sigma_mg, r'$\sigma_n=mg$'), (sigma_imp, r'$\sigma_n=50$ N')]):
        t, x, v, Fs, Ff = coulomb_sigma_n(0.6, 0.4, sn, m, k, v_load, dt, duration)
        axes[0,col].plot(t, Fs, 'b-', lw=0.8, label='$F_{spring}$')
        axes[0,col].plot(t, Ff, 'r-', lw=0.8, alpha=0.7, label='$F_{fric}$')
        axes[0,col].set_ylabel('Force (N)'); axes[0,col].set_title(f'Coulomb — {title}')
        axes[0,col].legend(fontsize=7); axes[0,col].grid(True, alpha=0.3)

    # --- RSF Scholz ---
    print("  RSF Scholz...")
    mu0=0.6; a_r=0.01; b_r=0.015; v0_r=1e-6; Dc_r=1e-5
    for col, (sn, title) in enumerate([(sigma_mg, r'$\sigma_n=mg$'), (sigma_imp, r'$\sigma_n=50$ N')]):
        t, x, v, Fs, Ff, th = rsf_scholz_euler(mu0, a_r, b_r, v0_r, Dc_r, sn, m, k, v_load, dt, duration)
        axes[1,col].plot(t, Fs, 'b-', lw=0.8, label='$F_{spring}$')
        axes[1,col].plot(t, Ff, 'r-', lw=0.8, alpha=0.7, label='$F_{fric}$')
        axes[1,col].set_ylabel('Force (N)'); axes[1,col].set_title(f'RSF Scholz — {title}')
        axes[1,col].legend(fontsize=7); axes[1,col].grid(True, alpha=0.3)

    # --- RK45 Erickson ---
    print("  RK45 Erickson...")
    M_rk=1; a_rk=0.008; b_rk=0.012; v0_rk=1e-3; Dc_rk=1e-5; k_rk=150; mu0_rk=0.6; vs=1e-3
    dur_rk = 15
    t5,x5,v5,_ = simulate_rk45_sigma_n(mu0_rk,a_rk,b_rk,v0_rk,vs,Dc_rk,M_rk,k_rk,M_rk*g,dur_rk,h_max=0.5)
    axes[2,0].plot(t5, v5, color='darkorange', lw=0.8)
    axes[2,0].set_ylabel('v (m/s)'); axes[2,0].set_title(r'RK45 — $\sigma_n=mg$')
    axes[2,0].set_xlabel('Time (s)'); axes[2,0].grid(True, alpha=0.3)
    
    alpha = sigma_imp/(M_rk*g)
    k_imp = k_rk * alpha
    t6,x6,v6,_ = simulate_rk45_sigma_n(mu0_rk,a_rk,b_rk,v0_rk,vs,Dc_rk,M_rk,k_imp,sigma_imp,dur_rk,h_max=0.5)
    axes[2,1].plot(t6, v6, color='darkorange', lw=0.8)
    axes[2,1].set_ylabel('v (m/s)')
    axes[2,1].set_title(rf'RK45 — $\sigma_n=50$ N, $k={k_imp:.0f}$')
    axes[2,1].set_xlabel('Time (s)'); axes[2,1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('code1_sigma_n.png', dpi=150)
    plt.close(fig); gc.collect()
    print("  Done. (Saved: code1_sigma_n.png)\n")
run_code1()

  CODE 1 : Comparaison sigma_n = m*g  vs  sigma_n = cte

  Coulomb...
  RSF Scholz...
  RK45 Erickson...
  72,932 steps (3,308 rejected)
  420,122 steps (20,153 rejected)
  Done. (Saved: code1_sigma_n.png)

